# 08 · Multi-Head Attention

**Belgedeki bölüm:** Multi-Head Attention

- Tek bir dikkat başı yerine birden fazla baş: her biri farklı bir ilişkiye bakar (özne–fiil, nesne–fiil …).
- Wq, Wk, Wv baş sayısına bölünür: d_model = 768, 96 baş → her baş 768 / 96 = 8 boyut.
- Başların çıktıları yan yana birleştirilir (concat) ve **Wo** ile çarpılır → yine d_model boyutu.

In [ ]:
import torch
from usta_embedding import UstaEmbedding
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united"

tokens = u_tokenizer.encode(prompt)

# UstaEmbedding = nn.Embedding + RoPE pozisyon bilgisi (04_Positional_Encoding, usta_embedding.py)
torch.manual_seed(1)
u_embedding = UstaEmbedding(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_embedding(tokens)
sentence_meanings.shape

### Elle: her baş ayrı bir UstaCausalAttention
Her baş embedding'in bir parçasıyla (4 / 2 = 2 boyut) çalışır; çıktılar birleştirilip Wo ile karıştırılır.

In [ ]:
import torch.nn as nn
from usta_causal_attention import UstaCausalAttention

class UstaMultiHeadAttentionManual(nn.Module):
  def __init__(self, embedding_dim, context_length, num_heads, dropout_rate = 0):
    super().__init__()
    head_dim = embedding_dim // num_heads  # her baş embedding'in bir parçasıyla çalışır

    self.heads = nn.ModuleList(
      [UstaCausalAttention(embedding_dim, head_dim, context_length, dropout_rate) for _ in range(num_heads)]
    )
    self.out_projection = nn.Linear(embedding_dim, embedding_dim)  # Wo

  def forward(self, x):
    attention_outs = []
    for head in self.heads:
      head_out = head(x)
      attention_outs.append(head_out)

    concatenated = torch.cat(attention_outs, dim=1)  # concat: başlar yan yana
    return self.out_projection(concatenated)

In [ ]:
torch.manual_seed(1)
multi_head_attention = UstaMultiHeadAttentionManual(embedding_dim=4, context_length=32, num_heads=2)

out = multi_head_attention(sentence_meanings)
out.shape, out

### Hazır modül: UstaMultiHeadAttention
PyTorch'un `nn.MultiheadAttention` katmanı + causal maske (`usta_multi_head_attention.py`).

In [ ]:
from usta_multi_head_attention import UstaMultiHeadAttention

torch.manual_seed(1)
mha = UstaMultiHeadAttention(embedding_dim=4, output_dim=4, context_length=32, num_heads=2)

out = mha(sentence_meanings)
out.shape

In [ ]:
mha

### Belgedeki boyutlarla: d_model = 768, 96 baş, 2 token

In [ ]:
x = torch.randn(2, 768)  # 2 token, her biri 768 boyutlu

mha_768 = nn.MultiheadAttention(embed_dim=768, num_heads=96)
print("baş başına boyut:", 768 // 96)

out, weights = mha_768(x, x, x)
print("çıktı:", out.shape)            # 2 x 768: boyut başta ve sonda aynı
print("Wo:", mha_768.out_proj.weight.shape)  # 768 x 768